In [2]:
import ee
import datetime
import pandas as pd
import requests
import rasterio
import os
from math import radians, sin, cos, atan2, sqrt

In [3]:
file_path = '/net/fs06/d3/rzhuang/TROPOMI_world/data/power_plant_location/filtered_power_plants.csv'
emission_df = pd.read_csv(file_path)
# emission_df.dropna(subset=["NOx Mass (short tons)"], inplace=True)
# file_path = '../data/facility-attributes.csv'
# information_df = pd.read_csv(file_path)

In [4]:
emission_df.sort_values(by=['nox_emis_ty'], ascending=False, inplace=True)

In [5]:
# Using pandas cumsum to accumulate NOx mass
emission_df["cumulative_NOx_sum"] = emission_df["nox_emis_ty"].cumsum()
emission_df["cumulative_CO2_sum"] = emission_df["co2_emis_ty"].cumsum()
# Calculate the percentage of NOx mass
emission_df["NOx_percentage"] = emission_df["cumulative_NOx_sum"] / emission_df["nox_emis_ty"].sum()
# Calculate the percentage of CO2 mass
emission_df["CO2_percentage"] = emission_df["cumulative_CO2_sum"] / emission_df["co2_emis_ty"].sum()

In [6]:
emission_df[:6000]

,ID,ISO3,fuel,longitude,latitude,co2_emis_ty,nox_emis_ty,sox_emis_ty,co_emis_ty,ch4_emis_ty,ID_MonthFact,ID_WeekFact,ID_HourFact,ID_VertProf,cumulative_NOx_sum,cumulative_CO2_sum,NOx_percentage,CO2_percentage
0,CoCO2_10706,RUS,natural gas,73.488900,61.279400,2.563959e+07,44548.165635,1283.673460,11549.626854,457.033662,FM_336,FW_275,FH_260,VP_09598,4.454817e+04,2.563959e+07,0.004323,0.002844
1,CoCO2_15522,POL,coal,19.326467,51.265983,3.840000e+07,30100.000000,45100.000000,26300.000000,99.999000,FM_139,FW_104,FH_095,VP_07680,7.464817e+04,6.403959e+07,0.007244,0.007104
2,CoCO2_11814,USA,coal,-78.675759,41.491466,6.542279e+05,29651.599383,0.000000,7229.197083,1.856072,FM_267,FW_222,FH_210,VP_02695,1.042998e+05,6.469382e+07,0.010122,0.007176
3,CoCO2_10698,RUS,natural gas,37.687900,55.916200,1.645677e+07,28593.238856,823.925774,7413.127670,293.347043,FM_336,FW_275,FH_260,VP_01932,1.328930e+05,8.115059e+07,0.012896,0.009002
4,CoCO2_10835,SAU,oil,39.555858,20.627494,1.271810e+07,28471.290029,87519.805057,4957.941986,508.044048,FM_353,FW_286,FH_271,VP_09123,1.613643e+05,9.386869e+07,0.015659,0.010413
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,CoCO2_14482,VEN,natural gas,-63.467804,9.700796,2.260849e+05,219.353257,11.319184,123.962320,4.030034,FM_357,FW_291,FH_276,VP_02929,9.903482e+06,8.249701e+09,0.961059,0.915124
5996,CoCO2_11965,USA,coal,-91.167500,43.335900,1.043772e+06,219.286945,274.616711,97.060516,10.773472,FM_197,FW_152,FH_140,VP_05883,9.903701e+06,8.250745e+09,0.961081,0.915240
5997,CoCO2_00144,ARG,oil,-71.252389,-41.158328,6.942176e+04,219.232440,217.370423,44.571129,2.758579,FM_006,FW_003,FH_276,VP_00769,9.903921e+06,8.250814e+09,0.961102,0.915247
5999,CoCO2_00137,ARG,oil,-57.824723,-38.247258,6.942176e+04,219.232440,217.370423,44.571129,2.758579,FM_006,FW_003,FH_276,VP_06571,9.904140e+06,8.250884e+09,0.961123,0.915255


In [7]:
emission_df.to_csv('/net/fs06/d3/rzhuang/TROPOMI_world/data/power_plant_location/filtered_power_plants.csv', index=False)


In [8]:
emission_df

,ID,ISO3,fuel,longitude,latitude,co2_emis_ty,nox_emis_ty,sox_emis_ty,co_emis_ty,ch4_emis_ty,ID_MonthFact,ID_WeekFact,ID_HourFact,ID_VertProf,cumulative_NOx_sum,cumulative_CO2_sum,NOx_percentage,CO2_percentage
0,CoCO2_10706,RUS,natural gas,73.488900,61.279400,2.563959e+07,44548.165635,1283.673460,11549.626854,457.033662,FM_336,FW_275,FH_260,VP_09598,4.454817e+04,2.563959e+07,0.004323,0.002844
1,CoCO2_15522,POL,coal,19.326467,51.265983,3.840000e+07,30100.000000,45100.000000,26300.000000,99.999000,FM_139,FW_104,FH_095,VP_07680,7.464817e+04,6.403959e+07,0.007244,0.007104
2,CoCO2_11814,USA,coal,-78.675759,41.491466,6.542279e+05,29651.599383,0.000000,7229.197083,1.856072,FM_267,FW_222,FH_210,VP_02695,1.042998e+05,6.469382e+07,0.010122,0.007176
3,CoCO2_10698,RUS,natural gas,37.687900,55.916200,1.645677e+07,28593.238856,823.925774,7413.127670,293.347043,FM_336,FW_275,FH_260,VP_01932,1.328930e+05,8.115059e+07,0.012896,0.009002
4,CoCO2_10835,SAU,oil,39.555858,20.627494,1.271810e+07,28471.290029,87519.805057,4957.941986,508.044048,FM_353,FW_286,FH_271,VP_09123,1.613643e+05,9.386869e+07,0.015659,0.010413
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13126,CoCO2_13067,USA,natural gas,-96.862100,42.269100,1.744443e+01,0.000000,0.000000,0.000000,0.000987,FM_243,FW_198,FH_186,VP_10772,1.030475e+07,9.014842e+09,1.000000,0.999999
13125,CoCO2_13859,USA,oil,-162.286322,63.521047,2.461664e+03,0.000000,0.000000,0.000000,0.099684,FM_345,FW_277,FH_262,VP_09494,1.030475e+07,9.014844e+09,1.000000,1.000000
13124,CoCO2_13863,USA,oil,-165.108575,60.530142,2.199927e+03,0.000000,0.000000,0.000000,0.089085,FM_345,FW_277,FH_262,VP_10107,1.030475e+07,9.014846e+09,1.000000,1.000000
13123,CoCO2_12322,USA,natural gas,-121.899969,37.378422,0.000000e+00,0.000000,0.000000,0.000000,0.003298,FM_180,FW_135,FH_123,VP_12012,1.030475e+07,9.014846e+09,1.000000,1.000000


In [7]:
df = pd.read_csv('../data/coco2_ps_catalogue_v2.0.csv')

In [ ]:
#!/usr/bin/env python3
import os, re, sys

def extract_urls(script_file):
    with open(script_file) as f:
        content = f.read()
    m = re.search(r"<<'EDSCEOF'\s*(.*?)\s*EDSCEOF", content, re.DOTALL)
    return m.group(1).strip().splitlines() if m else []

def get_filename(url):
    return url.rsplit('/', 1)[-1].split('?', 1)[0]

def main():
    script_file = '/net/fs06/d3/rzhuang/TROPOMI_world/data/TROPOMI_2018_data/download_parallel.sh'
    # Assign the directory to check; default is current directory
    directory = '/net/fs06/d3/rzhuang/TROPOMI_world/data/TROPOMI_2018_data'
    
    urls = extract_urls(script_file)
    missing = [url for url in urls
               if not os.path.isfile(os.path.join(directory, get_filename(url)))]
    
    if missing:
        print("Missing files in directory {}:".format(directory))
        for url in missing:
            print(url)
    else:
        print("All files are present in directory {}.".format(directory))

if __name__ == '__main__':
    main()

In [30]:
import pandas as pd
file_path = '../data/valid_tropomi_emissions_6000.csv'
emission_df = pd.read_csv(file_path)

/tmp/ipykernel_3037659/3269735481.py:3: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  emission_df = pd.read_csv(file_path)


In [31]:
emission_df.iloc[8021]

location                                             Site_2365
latitude                                             40.846439
longitude                                             8.307105
utc_time                           2018-05-04T13:32:24.372000Z
wind_u                                               -4.138534
wind_v                                               -3.395302
file_path    ../data/TROPOMI_2018_data/S5P_RPRO_L2__NO2____...
Name: 8021, dtype: object

In [24]:
emission_df.iloc[8021]

location                                             Site_3319
latitude                                             37.035362
longitude                                            27.900592
utc_time                           2018-05-01T11:05:46.799000Z
wind_u                                              0.20907864
wind_v                                               1.3255736
file_path    ../data/TROPOMI_2018_data/S5P_RPRO_L2__NO2____...
Name: 8021, dtype: object

In [8]:
df

,ID,ISO3,fuel,longitude,latitude,co2_emis_ty,nox_emis_ty,sox_emis_ty,co_emis_ty,ch4_emis_ty,ID_MonthFact,ID_WeekFact,ID_HourFact,ID_VertProf,cumulative_NOx_sum,cumulative_CO2_sum,NOx_percentage,CO2_percentage
0,CoCO2_10706,RUS,natural gas,73.488900,61.279400,2.563959e+07,44548.165635,1283.673460,11549.626854,457.033662,FM_336,FW_275,FH_260,VP_09598,4.454817e+04,2.563959e+07,0.002733,0.001898
1,CoCO2_15522,POL,coal,19.326467,51.265983,3.840000e+07,30100.000000,45100.000000,26300.000000,99.999000,FM_139,FW_104,FH_095,VP_07680,7.464817e+04,6.403959e+07,0.004579,0.004740
2,CoCO2_11814,USA,coal,-78.675759,41.491466,6.542279e+05,29651.599383,0.000000,7229.197083,1.856072,FM_267,FW_222,FH_210,VP_02695,1.042998e+05,6.469382e+07,0.006398,0.004788
3,CoCO2_10698,RUS,natural gas,37.687900,55.916200,1.645677e+07,28593.238856,823.925774,7413.127670,293.347043,FM_336,FW_275,FH_260,VP_01932,1.328930e+05,8.115059e+07,0.008152,0.006006
4,CoCO2_10835,SAU,oil,39.555858,20.627494,1.271810e+07,28471.290029,87519.805057,4957.941986,508.044048,FM_353,FW_286,FH_271,VP_09123,1.613643e+05,9.386869e+07,0.009898,0.006948
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16456,CoCO2_12368,USA,natural gas,-119.236667,35.734167,0.000000e+00,0.000000,0.000000,0.000000,0.063404,FM_180,FW_135,FH_123,VP_12012,1.630222e+07,1.351108e+10,1.000000,1.000000
16457,CoCO2_12359,USA,natural gas,-118.151910,34.081450,0.000000e+00,0.000000,0.000000,0.000000,0.004587,FM_180,FW_135,FH_123,VP_12012,1.630222e+07,1.351108e+10,1.000000,1.000000
16458,CoCO2_12357,USA,natural gas,-122.284003,37.832860,0.000000e+00,0.000000,0.000000,0.000000,0.055025,FM_180,FW_135,FH_123,VP_12012,1.630222e+07,1.351108e+10,1.000000,1.000000
16459,CoCO2_13702,USA,oil,-162.263717,64.616558,9.705017e+02,0.000000,0.000000,0.000000,0.039300,FM_345,FW_277,FH_262,VP_03014,1.630222e+07,1.351108e+10,1.000000,1.000000


In [ ]:
import pandas as pd
from haversine import haversine, Unit
import numpy as np # For NaN handling if needed

locations_df = pd.read_csv('../data/coco2_ps_catalogue_v2.0.csv')

# --- Configuration ---
# !! IMPORTANT: Adjust these column names if they are different in your DataFrame !!
lat_col = 'latitude'        # Name of the latitude column
lon_col = 'longitude'       # Name of the longitude column
emission_col = 'nox_emis_ty' # Name of the annual emissions column
# If you have a unique ID column, you might use it, but the DataFrame index works too.

# --- Load your DataFrame ---
# Assuming 'locations_df' is already loaded.
# Example loading (replace with your actual loading code):
# locations_df = pd.read_csv('your_power_plant_data.csv')

# --- Data Validation (Optional but Recommended) ---
# Ensure coordinate and emission columns are numeric
try:
    locations_df[lat_col] = pd.to_numeric(locations_df[lat_col])
    locations_df[lon_col] = pd.to_numeric(locations_df[lon_col])
    locations_df[emission_col] = pd.to_numeric(locations_df[emission_col])
except KeyError as e:
    print(f"Error: Column '{e}' not found. Please check your column names.")
    exit()
except ValueError:
    print(f"Error: Columns '{lat_col}', '{lon_col}', or '{emission_col}' contain non-numeric data.")
    # Consider adding error handling like errors='coerce' in pd.to_numeric
    # locations_df[lat_col] = pd.to_numeric(locations_df[lat_col], errors='coerce')
    # locations_df = locations_df.dropna(subset=[lat_col, lon_col, emission_col]) # Drop rows with conversion errors
    exit()

# Drop rows where essential data is missing
locations_df = locations_df.dropna(subset=[lat_col, lon_col, emission_col])
locations_df = locations_df.reset_index(drop=True) # Ensure index is continuous if rows were dropped


# --- Define Radii ---
radii_km = [20, 50, 100]

# --- Function to Calculate Stats for One Plant ---
def calculate_nearby_stats(reference_plant_row, all_plants_df, lat_col, lon_col, emission_col, radii):
    """
    Calculates count, total emissions, and percentage for nearby plants
    relative to a single reference plant.

    Args:
        reference_plant_row (pd.Series): The row of the reference plant.
        all_plants_df (pd.DataFrame): The entire DataFrame of plants.
        lat_col (str): Latitude column name.
        lon_col (str): Longitude column name.
        emission_col (str): Emission column name.
        radii (list): List of radii in kilometers to check.

    Returns:
        pd.Series: A series containing the calculated stats for the reference plant.
    """
    results = {}
    ref_coords = (reference_plant_row[lat_col], reference_plant_row[lon_col])
    ref_emission = reference_plant_row[emission_col]
    ref_index = reference_plant_row.name # Get the index of the reference plant

    # Calculate distances from the reference plant to all other plants
    # Avoid recalculating distance to self unnecessarily, but include self in sums later
    distances = all_plants_df.apply(
        lambda plant: haversine(ref_coords, (plant[lat_col], plant[lon_col]), unit=Unit.KILOMETERS),
        axis=1
    )

    for radius in radii:
        # Find mask for ALL plants within the radius (including the reference plant)
        within_radius_mask = distances <= radius

        # Get the DataFrame subset of plants within the radius
        nearby_plants_df = all_plants_df[within_radius_mask]

        # 1. Count: Number of OTHER plants within the radius
        # Exclude the reference plant itself from the count
        # The count is the number of True values in the mask, minus 1 (for the self-comparison if applicable)
        # Safer way: explicitly check index
        count_nearby = nearby_plants_df[nearby_plants_df.index != ref_index].shape[0]

        # 2. Total Emissions: Sum of emissions for ALL plants within the radius (including self)
        total_emissions_nearby = nearby_plants_df[emission_col].sum()

        # 3. Percentage: Reference plant's emission / total emissions nearby
        if total_emissions_nearby > 0 and not pd.isna(ref_emission):
             percentage = (ref_emission / total_emissions_nearby) * 100
        elif total_emissions_nearby == 0 and ref_emission == 0:
             percentage = 100.0 # Or 0.0 or np.nan, depending on how you define this edge case
        else:
             percentage = np.nan # Use NaN for undefined cases (e.g., division by zero, or ref_emission is NaN)


        results[f'nearby_plants_count_{radius}km'] = count_nearby
        results[f'total_emission_{radius}km'] = total_emissions_nearby
        results[f'percentage_emission_{radius}km'] = percentage

    return pd.Series(results)

# --- Apply the function to each row ---
print("Calculating nearby statistics... (This may take time for large datasets)")

# Use df.apply to iterate through each row (plant) and calculate its stats
# Pass the full dataframe `locations_df` and other args needed by the function
stats_results = locations_df.apply(
    calculate_nearby_stats,
    axis=1, # Apply function row-wise
    args=(locations_df, lat_col, lon_col, emission_col, radii_km) # Additional arguments for the function
)

# --- Combine results with the original DataFrame ---
locations_df = pd.concat([locations_df, stats_results], axis=1)

print("Calculations complete.")

# --- Display Results (Example) ---
print("\nDataFrame with added statistics:")
# Show relevant columns, adjust as needed
columns_to_show = [lat_col, lon_col, emission_col] + list(stats_results.columns)
print(locations_df[columns_to_show].head())

# You can now save or further analyze locations_df
locations_df.to_csv('power_plants_with_nearby_stats.csv', index=False)

Calculating nearby statistics... (This may take time for large datasets)
